CHECK AVAILABLE MODELS

In [1]:
import os
import torch
from pathlib import Path

# --- 1. Check GPU Hardware & VRAM ---
print(" GPU HARDWARE DIAGNOSTIC")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    allocated_mem = torch.cuda.memory_allocated(0) / (1024**3)
    free_mem = total_mem - allocated_mem
    print(f"GPU Model        : {gpu_name}")
    print(f"Total VRAM       : {total_mem:.2f} GB")
    print(f"Available VRAM   : {free_mem:.2f} GB")
else:
    print(" No CUDA GPU detected. Running on CPU.")

# --- 2. Check Hugging Face Local Model Cache ---
print("\n" + "=" * 50)
print(" LOCAL HUGGING FACE CACHE")
print("=" * 50)
hf_cache = Path.home() / ".cache" / "huggingface" / "hub"
if hf_cache.exists():
    cached_models = [d.name.replace("models--", "").replace("--", "/") 
                     for d in hf_cache.iterdir() if d.is_dir() and d.name.startswith("models--")]
    if cached_models:
        for m in cached_models:
            print(f" • {m}")
    else:
        print("No Hugging Face models found in local cache.")
else:
    print("Hugging Face cache directory not found.")

# --- 3. Check Local Ollama Models (if installed) ---
print(" LOCAL OLLAMA MODELS")
try:
    import subprocess
    res = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=5)
    print(res.stdout if res.stdout else "Ollama is installed but no models listed.")
except Exception:
    print("Ollama CLI not running or not found in environment.")

 GPU HARDWARE DIAGNOSTIC
GPU Model        : NVIDIA L40S
Total VRAM       : 44.39 GB
Available VRAM   : 44.39 GB

 LOCAL HUGGING FACE CACHE
 • BAAI/bge-large-en-v1.5
 • BAAI/bge-m3
 • NTQAI/pedestrian_gender_recognition
 LOCAL OLLAMA MODELS
NAME                                                            ID              SIZE      MODIFIED     
qwen3:4b-instruct                                               4b9a68b70468    2.5 GB    6 days ago      
qwen3:0.6b                                                      7df6b6e09427    522 MB    13 days ago     
bge-m3:latest                                                   790764642607    1.2 GB    13 days ago     
qwen3:1.7b                                                      3b6a3cd17b97    1.4 GB    4 weeks ago     
qwen3:14b                                                       bdbd181c33f2    9.3 GB    7 weeks ago     
Qwen3-4B-Instruct-2507-SOAP-subash-GGUF:latest                  4d0d3b2498dc    2.5 GB    2 months ago    
hf.co/rakuzai/

IMPORT LIBRARIES

In [2]:
import os
import re
import json
import time
import pandas as pd
import kagglehub
from ollama import Client

/home/jupyter-user/gender-classification-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DATASET SAMPLING

In [4]:
# Initialize Ollama Client
ollama_client = Client(host="http://localhost:11434")

# Selected Model
OLLAMA_MODEL = "qwen3:14b"  

print(" Loading Dataset (CSV Text Version)...")
dataset_path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")
csv_file = os.path.join(dataset_path, "Resume", "Resume.csv")

df = pd.read_csv(csv_file)

target_categories = ['ACCOUNTANT', 'AGRICULTURE', 'AVIATION', 'BANKING', 'FINANCE', 'HR', 'INFORMATION-TECHNOLOGY']
sample_frames = []

for cat in target_categories:
    cat_subset = df[df['Category'].str.upper() == cat.upper()]
    if len(cat_subset) >= 3:
        sample_frames.append(cat_subset.sample(n=3, random_state=42))
    else:
        sample_frames.append(cat_subset)

sample_df = pd.concat(sample_frames).reset_index(drop=True)

manifest = []
for idx, row in sample_df.iterrows():
    manifest.append({
        "id": f"CV_{idx+1:02d}",
        "category": row['Category'],
        "text": row['Resume_str']
    })

print(f" Loaded {len(manifest)} CVs (3 per role across {len(target_categories)} categories).")

 Loading Dataset (CSV Text Version)...
 Loaded 21 CVs (3 per role across 7 categories).


RULE-BASED (REGEX) EXTRACTION 

In [5]:
CATEGORY_SKILLS = {
    'ACCOUNTANT': [
        'accounting', 'quickbooks', 'taxation', 'auditing', 'bookkeeping', 
        'payroll', 'reconciliation', 'gaap', 'general ledger', 'accounts payable'
    ],
    'AGRICULTURE': [
        'agronomy', 'crop management', 'irrigation', 'soil science', 'harvesting', 
        'livestock', 'pest control', 'fertilizer', 'farm machinery', 'horticulture'
    ],
    'AVIATION': [
        'flight operations', 'faa', 'avionics', 'aircraft maintenance', 'air traffic control', 
        'navigation', 'pilot', 'flight planning', 'safety compliance', 'ground operations'
    ],
    'BANKING': [
        'credit analysis', 'loan processing', 'risk management', 'underwriting', 
        'commercial banking', 'retail banking', 'aml', 'kyc', 'wealth management', 'cash handling'
    ],
    'FINANCE': [
        'financial modeling', 'forecasting', 'valuation', 'budgeting', 'portfolio management', 
        'excel', 'investment banking', 'financial reporting', 'cfa', 'variance analysis'
    ],
    'HR': [
        'recruitment', 'talent acquisition', 'onboarding', 'hris', 'employee relations', 
        'performance management', 'labor law', 'training and development', 'payroll', 'benefits administration'
    ],
    'INFORMATION-TECHNOLOGY': [
        'python', 'java', 'sql', 'aws', 'docker', 'kubernetes', 'linux', 
        'machine learning', 'networking', 'troubleshooting', 'cybersecurity', 'system administration'
    ]
}

# Flatten all skills into a single list for global extraction
ALL_TARGET_SKILLS = sorted(list({skill for skills in CATEGORY_SKILLS.values() for skill in skills}))

In [6]:
def extract_with_regex(cv_text):
    start_time = time.time()
    
    # 1. GPA Extraction
    gpa_match = re.search(r'(?i)(?:gpa|ipk)[\s:]*([0-4]\.\d{1,2})', cv_text)
    gpa = float(gpa_match.group(1)) if gpa_match else None
    
    # 2. Experience & Seniority
    exp_match = re.search(r'(\d+)\+?\s*(?:years?|yrs?)\s*(?:of\s*)?experience', cv_text, re.IGNORECASE)
    years = int(exp_match.group(1)) if exp_match else 0
    
    if re.search(r'(?i)\b(senior|lead|principal|director|head of|manager)\b', cv_text):
        seniority = "Senior"
    elif years >= 5:
        seniority = "Senior"
    elif years >= 2 or re.search(r'(?i)\b(associate|specialist|officer)\b', cv_text):
        seniority = "Mid-Level"
    elif years > 0 or re.search(r'(?i)\b(intern|junior|entry|assistant)\b', cv_text):
        seniority = "Entry-Level"
    else:
        seniority = "Unknown"
    
    # 3. Mandatory Skills Extraction across all 7 domains
    found_skills = []
    text_lower = cv_text.lower()
    for skill in ALL_TARGET_SKILLS:
        if re.search(rf'\b{re.escape(skill)}\b', text_lower):
            found_skills.append(skill.title())
            
    latency = time.time() - start_time
    
    return {
        "seniority": seniority,
        "gpa": gpa,
        "skills": found_skills
    }, latency

print(" Regex Engine Initialized.")

 Regex Engine Initialized.


LLM EXTRACTION ENGINE (VIA OLLAMA)

In [7]:
structured_system_prompt = """You are an expert ATS information extraction system.
Extract structured attributes from the resume.

Rules:
1. Seniority must be one of: "Entry-Level", "Mid-Level", "Senior", or "Unknown" based on work history and titles.
2. GPA must be a float between 0.0 and 4.0 if explicitly stated; otherwise null.
3. Skills must be a list of up to 10 key technical and domain skills found in the text.

Respond ONLY with a valid JSON object matching this schema:
{
  "seniority": "Entry-Level | Mid-Level | Senior | Unknown",
  "gpa": float or null,
  "skills": ["skill1", "skill2"]
}"""

def extract_with_llm(cv_text):
    start_time = time.time()
    
    try:
        response = ollama_client.chat(
            model=OLLAMA_MODEL,
            messages=[
                {"role": "system", "content": structured_system_prompt},
                {"role": "user", "content": f"Resume Text:\n{cv_text[:5000]}"}
            ],
            format="json",
            options={"temperature": 0.2}
        )
        raw_content = response['message']['content'].strip()
        data = json.loads(raw_content)
    except Exception as e:
        data = {"seniority": "Error", "gpa": None, "skills": [], "error": str(e)}
        
    latency = time.time() - start_time
    return data, latency

print(f" Local Ollama LLM Engine Ready ({OLLAMA_MODEL}).")

 Local Ollama LLM Engine Ready (qwen3:14b).


COMPARISON

In [8]:
print(f" Running benchmark on {len(manifest)} CVs...\n")
print(f"{'ID':<6} | {'Category':<22} | {'Engine':<6} | {'Time':<8} | {'Seniority':<12} | {'GPA':<5} | {'Skills Extracted'}")
print("=" * 105)

benchmark_records = []
regex_times, llm_times = [], []

for cv in manifest:
    # 1. Execute Regex
    reg_out, reg_t = extract_with_regex(cv['text'])
    regex_times.append(reg_t)
    
    # 2. Execute LLM
    llm_out, llm_t = extract_with_llm(cv['text'])
    llm_times.append(llm_t)
    
    benchmark_records.append({
        "id": cv['id'],
        "category": cv['category'],
        "regex_seniority": reg_out.get('seniority'),
        "llm_seniority": llm_out.get('seniority'),
        "regex_gpa": reg_out.get('gpa'),
        "llm_gpa": llm_out.get('gpa'),
        "regex_skills": reg_out.get('skills', []),
        "llm_skills": llm_out.get('skills', []),
        "regex_time": reg_t,
        "llm_time": llm_t
    })
    
    # Display Side-by-Side Result
    print(f"{cv['id']:<6} | {cv['category']:<22} | {'Regex':<6} | {reg_t:.4f}s  | {str(reg_out.get('seniority')):<12} | {str(reg_out.get('gpa')):<5} | {', '.join(reg_out.get('skills', [])[:3])}")
    print(f"{'':<6} | {'':<22} | {'LLM':<6} | {llm_t:.4f}s  | {str(llm_out.get('seniority')):<12} | {str(llm_out.get('gpa')):<5} | {', '.join(llm_out.get('skills', [])[:3])}")
    print("-" * 105)

# Summary Metrics
avg_reg_time = sum(regex_times) / len(regex_times)
avg_llm_time = sum(llm_times) / len(llm_times)
speedup = avg_llm_time / avg_reg_time if avg_reg_time > 0 else 0

print(" BENCHMARK SUMMARY")
print(f"Total CVs Evaluated   : {len(manifest)}")
print(f"Average Regex Latency : {avg_reg_time*1000:.2f} ms per CV ({avg_reg_time:.4f}s)")
print(f"Average LLM Latency   : {avg_llm_time:.2f} s per CV")
print(f"Speed Differential    : Regex is {speedup:.1f}x faster than LLM")

# Save benchmark results for analysis
os.makedirs("../data", exist_ok=True)
with open("../data/regex_vs_llm_benchmark.json", "w", encoding="utf-8") as f:
    json.dump(benchmark_records, f, indent=2)
print(" Results saved to ../data/regex_vs_llm_benchmark.json")

 Running benchmark on 21 CVs...

ID     | Category               | Engine | Time     | Seniority    | GPA   | Skills Extracted
CV_01  | ACCOUNTANT             | Regex  | 0.0102s  | Senior       | None  | Accounting, Accounts Payable, Excel
       |                        | LLM    | 19.5122s  | Senior       | None  | Microsoft Excel, QuickBooks, Abila MIP Fund Accounting Software
---------------------------------------------------------------------------------------------------------
CV_02  | ACCOUNTANT             | Regex  | 0.0071s  | Unknown      | None  | Accounting, Accounts Payable, Excel
       |                        | LLM    | 15.1936s  | Senior       | None  | Quickbooks, JD Edwards, MAS90
---------------------------------------------------------------------------------------------------------
CV_03  | ACCOUNTANT             | Regex  | 0.0091s  | Senior       | None  | Accounting, Accounts Payable, Auditing
       |                        | LLM    | 13.8290s  | Senior       |